# Stock price time series

Plot adjusted closing prices from `data/Filter_historical` using the reusable function below. The final cell saves one graph per stock to `outputs/stock_time_series`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# Support launching the notebook from the project or notebooks directory.
PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent
DATA_FOLDER = PROJECT_FOLDER / "data" / "Filter_historical"

In [ ]:
def plot_stock_time_series(ticker, data_folder=DATA_FOLDER, price_column="Adj Close"):
    """Plot one stock's local price history and return its Matplotlib axes.

    ticker: CSV filename stem, e.g. "AAPL" (case-insensitive).
    data_folder: Directory containing one CSV per stock.
    price_column: Numeric price column to plot; defaults to adjusted close.
    """
    ticker = ticker.strip().upper()
    stock = pd.read_csv(Path(data_folder) / f"{ticker}.csv")

    # Check if essential columns are missing
    required = {"Date", price_column}
    missing = required.difference(stock.columns)
    if missing:
        raise ValueError(f"{ticker}: missing columns {sorted(missing)}")

    # turn string to datetime and numerical.
    stock["Date"] = pd.to_datetime(stock["Date"], errors="raise")
    stock[price_column] = pd.to_numeric(stock[price_column], errors="raise")

    # Drop no price stocks.
    stock = stock.dropna(subset=["Date"]).sort_values("Date")
    if stock[price_column].dropna().empty:
        raise ValueError(f"{ticker}: no prices available to plot")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(stock["Date"], stock[price_column], linewidth=1.2)
    ax.set_title(f"{ticker} — {price_column} time series")
    ax.set_xlabel("Date")
    ax.set_ylabel(f"{price_column} (USD)")
    fig.tight_layout()
    return ax

In [3]:
GRAPH_FOLDER = PROJECT_FOLDER / "outputs" / "stock_time_series"
GRAPH_FOLDER.mkdir(parents=True, exist_ok=True)
stock_files = sorted(DATA_FOLDER.glob("*.csv"))
if not stock_files:
    raise FileNotFoundError(f"No stock CSV files found in {DATA_FOLDER}")

saved = 0
failed = []
for stock_file in stock_files:
    try:
        ax = plot_stock_time_series(stock_file.stem)
        ax.figure.savefig(GRAPH_FOLDER / f"{stock_file.stem}.png", dpi=150)
        saved += 1
    except Exception as error:
        failed.append((stock_file.stem, str(error)))
    finally:
        plt.close("all")

print(f"Saved {saved} of {len(stock_files)} stock graphs to {GRAPH_FOLDER}")
for ticker, error in failed:
    print(f"Failed: {ticker}: {error}")

Saved 604 of 616 stock graphs to /home/lenovo/projects/pairs_trading_project/outputs/stock_time_series
Failed: APC: APC: no prices available to plot
Failed: BBBY: BBBY: no prices available to plot
Failed: CAM: CAM: no prices available to plot
Failed: CSRA: CSRA: no prices available to plot
Failed: FB: FB: no prices available to plot
Failed: INFO: INFO: no prices available to plot
Failed: LB: LB: no prices available to plot
Failed: NFX: NFX: no prices available to plot
Failed: SBNY: SBNY: no prices available to plot
Failed: SPLS: SPLS: no prices available to plot
Failed: STI: STI: no prices available to plot
Failed: TE: TE: no prices available to plot
